In [1]:
import time
import numpy as np
from aeon.classification.hybrid import HIVECOTEV2
from aeon.datasets import load_classification

X_train, y_train = load_classification("SyntheticControl", split="train")
X_test, y_test = load_classification("SyntheticControl", split="test")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (300, 1, 60), Test: (300, 1, 60)


In [2]:
configs = [
    {"label": "1 CPU (default)",          "n_jobs": 1, "parallel_backend": None},
    {"label": "8 CPUs (threading)",        "n_jobs": 8, "parallel_backend": None},
    {"label": "8 CPUs (loky)",             "n_jobs": 8, "parallel_backend": "loky"},
]

results = []
for cfg in configs:
    kwargs = {"n_jobs": cfg["n_jobs"]}
    if cfg["parallel_backend"] is not None:
        kwargs["parallel_backend"] = cfg["parallel_backend"]

    clf = HIVECOTEV2(**kwargs, random_state=42)

    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    preds = clf.predict(X_test)
    predict_time = time.perf_counter() - t0

    acc = np.mean(preds == y_test)
    results.append({"config": cfg["label"], "train_s": train_time, "predict_s": predict_time, "accuracy": acc})
    print(f"{cfg['label']:30s}  train={train_time:.1f}s  predict={predict_time:.2f}s  acc={acc:.3f}")

1 CPU (default)                 train=418.5s  predict=271.56s  acc=1.000
8 CPUs (threading)              train=1034.8s  predict=734.69s  acc=1.000
8 CPUs (loky)                   train=1035.2s  predict=735.40s  acc=1.000


In [3]:
import pandas as pd
df = pd.DataFrame(results)
df["speedup_vs_1cpu"] = df["train_s"].iloc[0] / df["train_s"]
df

,config,train_s,predict_s,accuracy,speedup_vs_1cpu
0,1 CPU (default),418.524425,271.555112,1.0,1.000000
1,8 CPUs (threading),1034.810126,734.688369,1.0,0.404446
2,8 CPUs (loky),1035.198702,735.400018,1.0,0.404294
